# Notebook 03 — Training Loop and Metrics

In Notebook 02 we wrote every part of a training step by hand. That's great for
understanding, but tedious when you have to do it in every notebook. Here we
introduce the reusable helpers that live in `utils/`: `train_one_epoch`,
`evaluate`, `fit`, `MetricTracker`, and the plotting functions. We'll use them
to train a slightly bigger CNN on CIFAR-10, then run a classic overfitting
demonstration with and without weight decay.

## Learning goals
- Understand what `model.train()` / `model.eval()` and `torch.no_grad()` do.
- Use `utils.training.fit` to train, validate, checkpoint, and early-stop.
- Track accuracy / precision / recall / F1 with `utils.metrics.MetricTracker`.
- Visualize training dynamics with `utils.plotting.plot_curves` and
  `plot_confusion_matrix`.
- Observe overfitting on a small subset, then recover generalization with
  weight decay.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/03_training_loop_and_metrics.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device


## 1. The anatomy of a training loop

A professional training loop almost always contains the same pieces. Once you
can recognize them, reading someone else's code becomes much easier.

### `model.train()` vs `model.eval()`

Many layers behave differently at training and inference time:

- **Dropout** is active during training (randomly zeroing activations) but is a
  no-op at eval time.
- **BatchNorm** updates its running mean/variance during training; at eval it
  uses the stored values.

Calling `model.train()` sets every module in the network to training mode;
`model.eval()` does the opposite. **Forget this, and your evaluation numbers
will be wrong.**

### `torch.no_grad()`

During evaluation we don't need gradients. Wrapping the forward pass in
`with torch.no_grad():` does two good things:

1. Skips building the autograd graph -> much less memory.
2. Skips the associated bookkeeping -> faster.

### Device placement

Your model and your data must live on the same device. The pattern is:

```python
model.to(device)
for xb, yb in loader:
    xb = xb.to(device)
    yb = yb.to(device)
    ...
```

### `tqdm` progress bars

Long training runs deserve a progress bar. `tqdm.auto.tqdm` picks the right
backend (Jupyter widgets vs console) automatically.

Our `utils.training.fit` encapsulates all of the above -- let's use it.


## 2. Load CIFAR-10 and build loaders

CIFAR-10 has 50,000 training + 10,000 test 32x32 RGB images across 10 classes.
We apply the same preprocessing template as Notebook 02 (`ToImage` ->
`ToDtype(float32)` -> `Normalize`) and wrap everything in `DataLoader`s.


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from utils.env import data_dir, checkpoints_dir

DATA_DIR = data_dir()
CKPT_DIR = checkpoints_dir()
print("DATA_DIR:", DATA_DIR)
print("CKPT_DIR:", CKPT_DIR)

# CIFAR-10 per-channel mean/std.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

tfms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR_MEAN, CIFAR_STD),
])

class CIFARWrapper(Dataset):
    """Applies a torchvision transform to a CIFAR10 (PIL, int) dataset."""
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        img, lbl = self.base[idx]
        return self.transform(img), lbl

train_raw = CIFAR10(root=DATA_DIR, train=True,  download=True)
val_raw   = CIFAR10(root=DATA_DIR, train=False, download=True)
class_names = train_raw.classes

train_ds = CIFARWrapper(train_raw, tfms)
val_ds   = CIFARWrapper(val_raw,   tfms)
print("train:", len(train_ds), "  val:", len(val_ds), "  classes:", class_names)

BATCH = 128
pin = (device.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=256,   shuffle=False, num_workers=0, pin_memory=pin)
print("train batches:", len(train_loader), "  val batches:", len(val_loader))


## 3. A small CNN for CIFAR-10

Fashion-MNIST was grayscale 28x28. CIFAR-10 is RGB 32x32, so we need a slightly
bigger network. The design below uses four conv blocks and a BatchNorm after
each conv, which both speeds up training and smooths the loss surface.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    """Four-block ConvNet for 32x32 RGB inputs."""

    def __init__(self, num_classes: int = 10):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            block(3,   32),    # 32 -> 16
            block(32,  64),    # 16 -> 8
            block(64, 128),    #  8 -> 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")


## 4. Train with `utils.training.fit`

Now the payoff: instead of writing another inner loop, we import `fit` from
`utils/`. It handles train/val loops, history bookkeeping, checkpointing and
early stopping.

For CPU users: five epochs on full CIFAR-10 will take a few minutes. If you're
on GPU it'll be under a minute. Feel free to drop `EPOCHS` to `2` if you just
want to see the mechanics.


In [ ]:
import os
from torch import optim
from utils.training import fit

loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

ckpt_path = os.path.join(CKPT_DIR, "nb03_best.pt")
EPOCHS = 5

history = fit(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    device=device,
    scheduler=None,
    early_stopping_patience=None,
    checkpoint_path=ckpt_path,
    verbose=True,
)
print("\nBest val acc:", history["best_val_acc"], "at epoch", history["best_epoch"])
print("Checkpoint saved to:", ckpt_path)


## 5. Metrics beyond accuracy -- `MetricTracker`

Accuracy alone can be misleading, especially with class imbalance. `utils.metrics.MetricTracker`
wraps `torchmetrics` to track accuracy, precision, recall and F1 across a full
evaluation pass with the same `update` / `compute` / `reset` pattern used by
its underlying library.

We'll run it over the validation set once, using the freshly trained model.


In [ ]:
from utils.metrics import MetricTracker, classification_report_dict
from utils.training import evaluate

# One evaluation pass using our utility -- gives us loss, acc, and the raw preds
val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader, loss_fn, device)
print(f"val loss = {val_loss:.4f}   val acc = {val_acc:.4f}")

# And the richer metrics with torchmetrics-backed MetricTracker:
tracker = MetricTracker(num_classes=10, task="multiclass", device=str(device))
model.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        tracker.update(logits, yb)
print("MetricTracker ->", tracker.compute())


In [ ]:
# sklearn classification report via our thin wrapper -- per-class precision/recall/F1.
report = classification_report_dict(y_true, y_pred, class_names=class_names)
# Pretty-print just a few rows so the output is readable.
for name in class_names[:5]:
    r = report[name]
    print(f"{name:12s}  precision={r['precision']:.3f}  recall={r['recall']:.3f}  f1={r['f1-score']:.3f}  support={int(r['support'])}")
print("...")
print(f"macro avg    precision={report['macro avg']['precision']:.3f}"
      f"  recall={report['macro avg']['recall']:.3f}"
      f"  f1={report['macro avg']['f1-score']:.3f}")


## 6. Visualize training with `utils.plotting`

`plot_curves(history)` takes the dict returned by `fit` and draws loss +
accuracy side by side. `plot_confusion_matrix(cm, class_names)` wraps seaborn's
annotated heatmap, which is perfect for spotting classes that the model mixes
up.


In [ ]:
from sklearn.metrics import confusion_matrix
from utils.plotting import plot_curves, plot_confusion_matrix

plot_curves(history)

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, class_names=class_names, normalize=True)


## 7. Save and restore the best checkpoint

`fit` already saved the best-val-accuracy state dict to `checkpoints/nb03_best.pt`.
A checkpoint is just a serialized dict; we decide what goes into it. Below we
load it back into a fresh model instance and confirm that its validation
accuracy matches.


In [ ]:
fresh = SmallCNN().to(device)
ckpt = torch.load(ckpt_path, map_location=device)
fresh.load_state_dict(ckpt["model_state_dict"])
print("Checkpoint was saved at epoch", ckpt["epoch"], " with val_acc =", ckpt["val_acc"])

reload_loss, reload_acc, _, _ = evaluate(fresh, val_loader, loss_fn, device)
print(f"reloaded model val acc = {reload_acc:.4f}  (should match the best reported above)")


## 8. Overfitting demo: 500 samples, many epochs

A small training set + a reasonably expressive model is a recipe for
overfitting. Here's the textbook picture: train accuracy climbs to ~100% while
validation accuracy plateaus (or even drops slightly). Then we repeat the
experiment with `weight_decay=1e-2`, which is a mild L2 penalty on weights, and
see validation improve.


In [ ]:
# 500-sample "pathologically small" training set.
tiny_train_ds = Subset(train_ds, indices=list(range(500)))
tiny_train_loader = DataLoader(tiny_train_ds, batch_size=64, shuffle=True,
                               num_workers=0, pin_memory=pin)
print("tiny train size:", len(tiny_train_ds), "  batches:", len(tiny_train_loader))

OVERFIT_EPOCHS = 20


In [ ]:
# --- Run 1: NO weight decay (expect overfitting) ---
model_a = SmallCNN().to(device)
opt_a   = optim.Adam(model_a.parameters(), lr=1e-3, weight_decay=0.0)
hist_a  = fit(
    model_a, tiny_train_loader, val_loader, loss_fn, opt_a,
    epochs=OVERFIT_EPOCHS, device=device, verbose=False,
)
print(f"NO weight_decay: best val acc = {hist_a['best_val_acc']:.4f} at epoch {hist_a['best_epoch']+1}")

# --- Run 2: weight_decay=1e-2 ---
model_b = SmallCNN().to(device)
opt_b   = optim.Adam(model_b.parameters(), lr=1e-3, weight_decay=1e-2)
hist_b  = fit(
    model_b, tiny_train_loader, val_loader, loss_fn, opt_b,
    epochs=OVERFIT_EPOCHS, device=device, verbose=False,
)
print(f"weight_decay=1e-2: best val acc = {hist_b['best_val_acc']:.4f} at epoch {hist_b['best_epoch']+1}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (title, h) in zip(axes, [("No weight decay", hist_a), ("weight_decay=1e-2", hist_b)]):
    ax.plot(h["train_acc"], label="train", marker="o")
    ax.plot(h["val_acc"],   label="val",   marker="o")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()


You should see the left plot (no weight decay) with train accuracy racing
toward 1.0 while validation stalls -- that's the overfitting signature. The
right plot (with weight decay) keeps train accuracy more modest, which leaves
more room for the validation curve to rise. The exact numbers depend on the
random seed but the *shape* is robust.


## Key Takeaways

- Always call `model.train()` before training and `model.eval()` +
  `torch.no_grad()` for evaluation -- dropout and batchnorm depend on it.
- `utils.training.fit` wraps the full loop: training, evaluation, checkpointing
  and early stopping in a single call.
- Beyond accuracy, track precision / recall / F1 with `MetricTracker`. A
  confusion matrix is often more informative than a single scalar.
- `plot_curves` and `plot_confusion_matrix` make "is my model working?" a
  one-line question.
- Small training sets overfit quickly. Weight decay is a cheap, effective
  regularizer that pushes validation accuracy up.


## Exercises

1. **Early stopping.** Re-run the full 5-epoch training with
   `early_stopping_patience=2`. Does it stop early? Change `EPOCHS=20` and
   `patience=3` and observe.
2. **Learning-rate schedule.** Wrap the optimizer with
   `torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)` and
   pass it to `fit(..., scheduler=...)`. Does validation accuracy improve?
3. **Dropout ablation.** Remove the two `nn.Dropout` layers from `SmallCNN` and
   repeat the 500-sample overfitting experiment. How much worse does the
   no-weight-decay run get?
